# 01 · Conversation Overview
El inventario mostró que las conversaciones reales están principalmente en PDF. Este notebook cambia el flujo: primero extrae texto de los PDFs, inspecciona el formato real de Mantra y recién después define el parser canónico. No usa OCR ni modifica los archivos originales.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/ai_assistant
!git clone -q https://github.com/dinatalediego/ai_assistant.git /content/ai_assistant
%cd /content/ai_assistant
!pip -q install pandas pymupdf pyarrow
from pathlib import Path
import pandas as pd
ROOT=Path('/content/drive/MyDrive/mine_chatbot')
OUT=ROOT/'_analysis_outputs'
OUT.mkdir(exist_ok=True)
print('ROOT:', ROOT)

## 1. Confirmar que el dataset es PDF
El notebook anterior buscaba columnas tabulares. Eso no aplica a PDFs; por eso el contrato canónico aparecía vacío.

In [ ]:
inventory=pd.read_csv(OUT/'file_inventory.csv')
summary=(inventory.groupby('suffix',dropna=False).agg(archivos=('name','size'),bytes=('bytes','sum')).sort_values('archivos',ascending=False))
display(summary.head(20))
pdf_inventory=inventory[inventory['suffix'].eq('.pdf')].copy()
print('PDFs encontrados:', len(pdf_inventory))

## 2. Extraer texto de una muestra
Se usa PyMuPDF sobre texto seleccionable. Primero inspeccionamos una muestra pequeña para descubrir cómo Mantra representa remitente, fecha/hora y mensajes.

In [ ]:
from src.pdf_conversations import build_pdf_text_index
sample=build_pdf_text_index(ROOT,max_files=20,max_pages=10)
display(sample[['source_file','pages','text_chars','text_extractable','error']])
print('Con texto extraíble:', int(sample['text_extractable'].sum()), '/', len(sample))

In [ ]:
pd.set_option('display.max_colwidth', 4000)
for _,r in sample[sample['text_extractable']].head(5).iterrows():
    print('\n'+'='*100)
    print(r['source_file'])
    print('='*100)
    print(r['text_preview'][:4000])

## 3. Metadata candidata desde el nombre del PDF
Los nombres parecen contener secuencia, nombre del lead y un identificador numérico. Se conservan como candidatos y no se declaran equivalencias definitivas hasta validarlas.

In [ ]:
display(sample[['source_file','file_sequence_candidate','lead_name_candidate','lead_numeric_candidate']].head(20))

## 4. Perfil de extractabilidad
Antes de procesar miles de PDFs, medimos si contienen texto seleccionable. Si una fracción relevante no lo contiene, esos archivos quedan marcados para revisión separada; no se activa OCR automáticamente.

In [ ]:
profile=pd.Series({
    'sample_files':len(sample),
    'extractable_files':int(sample['text_extractable'].sum()),
    'non_extractable_files':int((~sample['text_extractable']).sum()),
    'median_text_chars':float(sample.loc[sample['text_extractable'],'text_chars'].median()) if sample['text_extractable'].any() else 0,
    'errors':int(sample['error'].notna().sum())
})
display(profile.to_frame('valor'))
sample.to_csv(OUT/'pdf_text_sample.csv',index=False)
print('Guardado:', OUT/'pdf_text_sample.csv')

## 5. Contrato canónico — pendiente de evidencia
El contrato objetivo sigue siendo `conversation_id`, `message_id`, `timestamp`, `sender`, `text`, `lead_id`, `project`, pero ahora se definirá a partir del patrón textual real observado en los PDFs. El siguiente paso es construir un parser específico de Mantra usando las cinco previsualizaciones anteriores, no inferir columnas que no existen.